# Customer Churn Prediction & Retention Analytics
**Author:** Betanapalli Lokesh (IIT ISM Dhanbad)

### Problem Overview
Customer churn is a critical metric for subscription-based businesses (e.g., Microsoft 365, Azure, cloud telecom services). 
Acquiring a new customer can cost 5-7x more than retaining an existing one. 

**Objective:**
1. Predict the probability of a customer churning (`Churn = Yes/No`).
2. Identify key behavioral and contract features that trigger customer attrition.
3. Evaluate models using **ROC-AUC** and **Recall** due to class imbalance.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

import warnings
warnings.filterwarnings('ignore')

## 1. Data Ingestion & Exploration (EDA)

In [2]:
# Load dataset
df = pd.read_csv('../data/telco_customer_churn.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

In [3]:
# Churn Distribution Check
churn_counts = df['Churn'].value_counts(normalize=True) * 100
print("Churn Distribution (%):")
print(churn_counts)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Churn', palette='pastel')
plt.title('Class Imbalance: Churn vs. Retained Customers')
plt.show()

> **Key Observation:** ~28% of customers churn, showing clear **class imbalance**. Raw accuracy is misleading; we must evaluate using **Recall** and **ROC-AUC**.

## 2. Feature Engineering & Preprocessing

In [4]:
# Drop identifier
df_clean = df.drop(columns=['customerID'])

# Map binary target
df_clean['Churn'] = df_clean['Churn'].map({'Yes': 1, 'No': 0})

# One-Hot Encoding for categorical features
cat_cols = df_clean.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)

X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

# Stratified Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature Scaling
scaler = StandardScaler()
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print(f"Training set: {X_train.shape[0]} samples, Test set: {X_test.shape[0]} samples")

## 3. Model Training & Comparison

In [5]:
# Baseline: Logistic Regression (balanced weights)
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Advanced: Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=150, max_depth=8, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate Probabilities
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("Logistic Regression ROC-AUC:", round(roc_auc_score(y_test, y_prob_lr), 4))
print("Random Forest ROC-AUC:      ", round(roc_auc_score(y_test, y_prob_rf), 4))

## 4. Evaluation & Visual Artifacts

In [6]:
# ROC Curve Plotting
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

plt.figure(figsize=(7, 5))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {roc_auc_score(y_test, y_prob_lr):.2f})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_score(y_test, y_prob_rf):.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [7]:
# Top Feature Importances
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(9, 5))
sns.barplot(x=importances.values, y=importances.index, palette='viridis')
plt.title('Top 10 Drivers of Customer Churn')
plt.xlabel('Importance Score')
plt.show()

## 5. Strategic Retention Takeaways
1. **Contract Type:** Month-to-month contracts have the highest risk of churn. Promoting 1-year or 2-year discounts directly protects retention.
2. **Tenure:** First 6-12 months are high vulnerability periods. Onboarding support must be prioritized.
3. **Add-on Services:** TechSupport and OnlineSecurity significantly increase customer retention.